In [12]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings,ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
load_dotenv()

True

In [18]:
def get_transcript(video_id):
    ytt_api = YouTubeTranscriptApi()

    transcript_list = ytt_api.list(video_id)


    print("Available transcripts:")
    for t in transcript_list:
        print(t.language_code, "-", t.language)

    try:
        transcript = transcript_list.find_transcript(
            ['en', 'en-US', 'hi']
        ).fetch()

    except:
        transcript = next(iter(transcript_list)).fetch()

    return transcript


video_id = "a3C1DMswClQ"

transcript = get_transcript(video_id)

full_text = " ".join([x.text for x in transcript])

print(full_text[:1000])

Available transcripts:
en - English (auto-generated)
backend is huge and if we start discussing every single component that could be part of it we will be stuck here for years so what we will do is we will only discuss those topics which are used in majority of the code bases let's say 90% of them with that in mind let's talk about HTTP protocol the medium through which our browsers talk to our servers either to send data or to receive data from it and as I said there are a lot of other ways and protocols which clients and servers use to communicate with each other and HTTP being one of the most used ones we will focus on that now there are two ideas which are at the heart of HTTP protocol the first one being tessness what does it mean tessness basically means it has no memory of past interactions so each HTTP request carries all the necessary information for the server to process it such as headers or URLs and methods which we will see in a bit and after the server responds it forgets

In [19]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([full_text])

In [20]:
len(chunks)

85

In [21]:
chunks[10]

Document(metadata={}, page_content="means the status code 200 means okay right and we have again some response headers after that there is a blank line and we have the response body so let's talk about HTTP headers first this is very important since it's the major part of the request here and the responses on a high level what we can see is headers are basically key value pairs key value pairs of different different parameters that is sent over request or received over a response that is the first definition and the next question is why do we need headers why not send the all these values in the you know the URL or we have this request body here why send why create a different section just for sending more information or more met data about the request or the response why create another level of abstraction and in order to understand that let's take a real life example we send Parcels right ciers and we receive them the address of about the phone number of the recipient and different d

In [23]:
embeddings = HuggingFaceEndpointEmbeddings(
    repo_id="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(chunks, embeddings)

In [24]:
vector_store.index_to_docstore_id

{0: 'e79832f1-950f-4e26-b398-18f062aae229',
 1: '2efcec9e-3787-4e73-a5a9-cd72568aac56',
 2: '26b26f08-fd81-48dc-aa51-ad53ca3f3ecf',
 3: '98a76b32-bde9-42e9-9e9b-715bca30e342',
 4: '5a04ccad-994a-4f6f-b88b-ea5c9b71e628',
 5: '9d465380-6471-46bb-9e3f-49528066fb8c',
 6: '1cde5f60-c592-4b88-8685-fe63e27281c8',
 7: 'abcf53ca-49ff-4094-b416-0abf75671b94',
 8: '4c7a9034-ba95-44b8-b611-b40c7a95fd60',
 9: '25baea5c-1665-4738-aa7b-471ab1920259',
 10: '0d725caa-61a0-4b4d-a214-fbb85639ad85',
 11: 'adcd81f5-78fb-47dc-8e82-120e1714a29f',
 12: 'c1eece34-163c-4c11-ac8a-02010ce747ad',
 13: 'dfadb5b6-675b-4452-bac2-dba86125c896',
 14: '244283c7-f593-4740-85a0-1c490372010a',
 15: 'db06120b-7243-435e-89fe-b98c0c5ea0e3',
 16: 'd07bd7bb-d4be-4691-a4e1-ca36d03ebfdd',
 17: 'a74799ae-d2c7-4a4a-ae92-068009c8d524',
 18: 'd80a2d26-28be-4d2c-bd10-bf3a14e85540',
 19: '028b0ddc-78d9-4d98-a031-70ef1d386628',
 20: '2f136d6e-25a6-40ec-89c1-55daafffaada',
 21: 'f5c89e58-9b91-456a-9f7f-854aa2a9ca82',
 22: 'df6989df-b3b7-

In [26]:
vector_store.get_by_ids(['d08315b9-50f7-4c46-a21d-0d23bf996661'])

[Document(id='d08315b9-50f7-4c46-a21d-0d23bf996661', metadata={}, page_content="cross origin request there are primarily two types of flows one is a simple request flow and another one is a pre-f flighter request flow first let's look at the simple request imagine our client our frontend is at the Domain example.com and our server is at the Domain api. example.com and we make a get request which looks something like like this this is how the flow looks like the client sends a request the browser automatically adds the origin header to indicate the origin of the request the request uses a simple method which are usually get post or head and then it reaches the server the server checks for the origin header against it C policy if the origin is allowed the server includes the access control allow origin header in the response and it sends the response response the server responds with the resource and includes the necessary course headers for example Access Control allow origin header and

In [27]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [28]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEndpointEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002A20CEC83A0>, search_kwargs={'k': 4})

In [29]:
retriever.invoke('what is https' )

[Document(id='5a04ccad-994a-4f6f-b88b-ea5c9b71e628', metadata={}, page_content='by the server and throughout our discussion we can go ahead and safely assume that HTTP and https are interchangeable because https to oversimplify it just a more secure version of HTTP but the underlying principles are the same with more security features like encryption and security certificates or TLS stuff like that which boarders too much into the network engineering domain also to send some kind of request or receive some kind of response first the client and the server need to establish some kind of connection mechanism right otherwise how are they going to communicate what is the medium of the communication and for that sgtp uses TCP TCP which is a protocol which is a transmission protocol essentially HTTP does not require the underlying transport protocol to be connection based it only requires it to be reliable and not lose message messages at minimum presenting an error in such cases and among th

In [43]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3.1-Terminus",
    task="conversational",
)
model = ChatHuggingFace(llm=llm)

In [31]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [32]:
question          = "what is cors"
retrieved_docs    = retriever.invoke(question)

In [33]:
retrieved_docs

[Document(id='d08315b9-50f7-4c46-a21d-0d23bf996661', metadata={}, page_content="cross origin request there are primarily two types of flows one is a simple request flow and another one is a pre-f flighter request flow first let's look at the simple request imagine our client our frontend is at the Domain example.com and our server is at the Domain api. example.com and we make a get request which looks something like like this this is how the flow looks like the client sends a request the browser automatically adds the origin header to indicate the origin of the request the request uses a simple method which are usually get post or head and then it reaches the server the server checks for the origin header against it C policy if the origin is allowed the server includes the access control allow origin header in the response and it sends the response response the server responds with the resource and includes the necessary course headers for example Access Control allow origin header and

In [34]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"cross origin request there are primarily two types of flows one is a simple request flow and another one is a pre-f flighter request flow first let's look at the simple request imagine our client our frontend is at the Domain example.com and our server is at the Domain api. example.com and we make a get request which looks something like like this this is how the flow looks like the client sends a request the browser automatically adds the origin header to indicate the origin of the request the request uses a simple method which are usually get post or head and then it reaches the server the server checks for the origin header against it C policy if the origin is allowed the server includes the access control allow origin header in the response and it sends the response response the server responds with the resource and includes the necessary course headers for example Access Control allow origin header and this is what the response looks like it has the appropriate course header\n\ns

In [35]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [38]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      cross origin request there are primarily two types of flows one is a simple request flow and another one is a pre-f flighter request flow first let's look at the simple request imagine our client our frontend is at the Domain example.com and our server is at the Domain api. example.com and we make a get request which looks something like like this this is how the flow looks like the client sends a request the browser automatically adds the origin header to indicate the origin of the request the request uses a simple method which are usually get post or head and then it reaches the server the server checks for the origin header against it C policy if the origin is allowed the server includes the access control allow origin header in the response and it sends the response response the server responds

In [47]:
answer = model.invoke(final_prompt)
print(answer.content)

Based solely on the provided transcript, CORS (Cross-Origin Resource Sharing) is a mechanism that involves flows for cross-origin requests. There are primarily two types: a simple request flow and a pre-flighted request flow. It involves the browser automatically adding an Origin header to a request and the server checking this against its policy. The server must include specific headers like `Access-Control-Allow-Origin` in its response for the browser to allow the frontend to receive and render the response. If the server does not include the necessary headers, the browser blocks the response with a CORS error.


In [48]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [49]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [50]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [52]:
parallel_chain.invoke('what is http')

{'context': "HTTP request flow there is a client and there is a server always the client typically is a web browser or an application which initiates the communication by sending a request to the server the client is responsible for for providing all information needed by the server such as the URL of the resource or the headers and everything and then there is a server which hosts resources like websites or apis or other content and waits for incoming requests from the clients when the server receives a request it processes it and sends back the appropriate response such as a web page or data or error message or a Json file or a text file or any other kind of content now the thing to remember here is HTTP protocol states that communication is always initiated by the client to get some kind of resp response by the server and throughout our discussion we can go ahead and safely assume that HTTP and https are interchangeable because https to oversimplify it just a more secure version of 

In [53]:
parser = StrOutputParser()

In [55]:
main_chain = parallel_chain | prompt | model | parser

In [56]:
main_chain.invoke('Can you summarize the video')

'Based on the transcript context, the video covers understanding HTTP concepts from first principles without looking at code, using tools like Burp Suite. It explains the flow of HTTP components, including how data streaming works with chunks and headers like content-type and connection keep-alive, as well as the CORS flow with pre-flight requests and simple requests. It also introduces HTTP response codes as a way to standardize request results.'

In [57]:
main_chain.invoke('how does cors help')

'CORS helps by allowing servers to specify who can access their resources and how in a cross-origin request, ensuring that browsers can safely make requests from one origin to another without being blocked by the same-origin policy.'